# nb_gold_populacao — Camada Gold População (Fabric)

**Fonte:** `silver_populacao`  
**Saída:** `gold_populacao_municipios`  
**Granularidade:** município × ano  
**Indicadores:** população · variação anual absoluta · variação anual (%)

In [ ]:
%run ./nb_utils_ibge

In [ ]:
from pyspark.sql.functions import col, round as spark_round, lag, when
from pyspark.sql.window import Window

## 1. Carregar Silver

In [ ]:
# Cluster mapping
cluster_rows = [
    (code, cluster)
    for cluster, codes in CLUSTERS.items()
    for code in codes
]
df_cluster = spark.createDataFrame(cluster_rows, ["id_municipio", "cluster"])

# silver_populacao — cast ano para int (armazenado como varchar no Fabric)
df_pop = spark.sql("""
    SELECT
        id_municipio,
        nome_municipio,
        CAST(ano AS INT) AS ano,
        valor            AS populacao
    FROM silver_populacao
    WHERE indicador = 'populacao_residente'
""")

print(f"[OK] silver_populacao: {df_pop.count()} registros · anos: ", end="")
print(df_pop.agg({"ano": "min"}).collect()[0][0], "–",
      df_pop.agg({"ano": "max"}).collect()[0][0])

## 2. Calcular variação anual

In [ ]:
# Window por município ordenado por ano — para calcular variação YoY
w_mun = Window.partitionBy("id_municipio").orderBy("ano")

df_gold = (
    df_pop
    .join(df_cluster, on="id_municipio", how="left")
    .withColumn("populacao_ano_anterior", lag("populacao", 1).over(w_mun))
    .withColumn(
        "variacao_abs",
        col("populacao") - col("populacao_ano_anterior")
    )
    .withColumn(
        "variacao_anual_pct",
        spark_round(
            when(
                col("populacao_ano_anterior") > 0,
                (col("populacao") - col("populacao_ano_anterior"))
                / col("populacao_ano_anterior") * 100
            ),
            2
        )
    )
    .select(
        "id_municipio", "nome_municipio", "cluster", "ano",
        "populacao", "variacao_abs", "variacao_anual_pct"
    )
)

assert df_gold.count() > 0, "[ERRO] gold_populacao_municipios vazio antes de gravar"
print(f"[OK] gold_populacao_municipios: {df_gold.count()} registros")
display(df_gold.orderBy("id_municipio", "ano").limit(20))

## 3. Gravar Gold

In [ ]:
save_delta(df_gold, "gold_populacao_municipios")
print("[OK] gold_populacao_municipios gravada")

# Spot check — Santos, Osasco, Mauá — últimos 3 anos
ano_max = df_gold.agg({"ano": "max"}).collect()[0][0]
display(
    df_gold
    .filter(
        col("id_municipio").isin(3548500, 3534401, 3529401) &
        (col("ano") >= ano_max - 2)
    )
    .orderBy("id_municipio", "ano")
)